In this notebook, I will analyse and answer:

#### “Given the demand patterns and shipping behavior, how should inventory be managed efficiently?”

This includes:

- deciding warehouse replenishment 
- preventing stockouts
- reducing storage cost
- which products are high-risk
- optimising inventory levels to align with operations and meet customer service

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import matplotlib.ticker as ticker

import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv("../data/raw/amazon_ecommerce_1M.csv")

In [4]:
df['purchase_date'] = pd.to_datetime(df['purchase_date'])

df['revenue'] = df['final_price']

df['return_flag'] = df['is_returned'].astype(int)

Since this dataset does not have inventory movement data.

- I define demand as: number of purchases per category per day

In [5]:
category_demand = (
    df.groupby(['purchase_date', 'category'])
    .size()
    .reset_index(name='demand')
)

In [10]:
category_demand.head(10)

,purchase_date,category,demand
0,2024-03-31,Beauty,293
1,2024-03-31,Clothing,273
2,2024-03-31,Electronics,272
3,2024-03-31,Home,292
4,2024-03-31,Sports,270
5,2024-04-01,Beauty,271
6,2024-04-01,Clothing,268
7,2024-04-01,Electronics,276
8,2024-04-01,Home,283
9,2024-04-01,Sports,282


### LEAD TIME Concept

- time it takes for inventory to be replenished and arrive at the customer

In [11]:
lead_time = df['shipping_time_days']

In [12]:
avg_lead_time = df['shipping_time_days'].mean()

print(f"Average Lead Time: {avg_lead_time:.2f} days")

Average Lead Time: 3.17 days


### SAFETY STOCK 

- Safety stock is extra inventory kept to prevent stockouts due to uncertainty

In [13]:
demand_std = category_demand.groupby('category')['demand'].std()

Calculating Safety Stock

- Z = service level factor (we assume 1.65 for ~95%)
- σd = demand standard deviation
- L = lead time

In [14]:
Z = 1.65

lead_time = df['shipping_time_days'].mean()

safety_stock = Z * demand_std * np.sqrt(lead_time)

safety_stock = safety_stock.reset_index()
safety_stock.columns = ['category', 'safety_stock']

Categories with:

- high volatility → need more safety stock
- stable demand → need less safety stock

In [15]:
safety_stock.head(10)

,category,safety_stock
0,Beauty,46.700234
1,Clothing,47.603337
2,Electronics,49.713824
3,Home,46.893111
4,Sports,48.235290
